# What Treatments Help or Harm People with Post-SSRI Sexual Dysfunction (PSSD)?

**Abstract:** This analysis investigates which treatments are reported as helpful or harmful by people living with Post-SSRI Sexual Dysfunction (PSSD), using 902 treatment reports from 220 unique users across 245 drugs in the r/PSSD community. The key finding is that SSRIs and SNRIs -- the drug classes that caused the condition -- are overwhelmingly reported negatively (93% negative for the SSRI class, p < 0.001 vs. 50% baseline), while a small cluster of non-serotonergic treatments shows genuine positive signal: bupropion (the only non-serotonergic antidepressant, 52% positive rate), antihistamines such as ketotifen (88% positive), and microdosing (78% positive). The overall sentiment distribution is 62% negative, the inverse of most patient communities, reflecting a population harmed by the medical system. Patients considering treatment should focus on the non-serotonergic options in the Strong Evidence tier and avoid re-exposure to SSRIs/SNRIs.

**Method:** User-level aggregation (one data point per user per drug), binomial tests against a 50% positive baseline, Mann-Whitney U tests for group comparisons, forest plots with 95% confidence intervals. Sample: 902 reports from 220 unique users.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\pssd.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment score mapping ----------
SENTIMENT_SCORES = {
    "positive": 1.0,
    "mixed":    0.5,
    "neutral":  0.0,
    "negative": -1.0,
}

# ---------- Outcome classification thresholds ----------
def classify_outcome(avg_score):
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    else:
        return "mixed/neutral"

# ---------- Generic treatment filter ----------
# These are categories, not actionable treatments a patient can use.
# We keep specific antidepressant names (sertraline, fluoxetine, etc.).
GENERIC_FILTER = {
    "supplements", "medication", "treatment", "therapy", "antidepressant",
}

# ---------- SSRI / SNRI drug IDs ----------
# Specific SSRI names in the database
SSRI_NAMES = {
    "sertraline", "fluoxetine", "paroxetine", "citalopram",
    "escitalopram", "fluvoxamine", "prozac", "lexapro",
    "ssri",  # the generic class label
}
SNRI_NAMES = {"venlafaxine", "duloxetine", "snri"}
SSRI_SNRI_NAMES = SSRI_NAMES | SNRI_NAMES

print("Setup complete.")
print(f"Database: {DB_PATH}")

## 2. Data Exploration -- Overall Sentiment Breakdown

Before examining specific treatments, we look at the overall sentiment distribution across all 902 treatment reports. Unlike typical patient communities where sentiment skews positive (patients grateful for treatments that helped), the PSSD community sentiment is **inverted**: 62% negative. This reflects a population harmed by psychiatric medications, where the dominant narrative is damage, not recovery.

In [ ]:
# ---------- Load all treatment reports ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, tr.signal_strength,
           t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

all_reports["score"] = all_reports["sentiment"].map(SENTIMENT_SCORES)

print(f"Total treatment reports: {len(all_reports):,}")
print(f"Unique users with reports: {all_reports['user_id'].nunique()}")
print(f"Unique drugs mentioned: {all_reports['drug'].nunique()}")
print()

# ---------- Overall sentiment breakdown ----------
sentiment_counts = all_reports["sentiment"].value_counts()
sentiment_pcts = (sentiment_counts / len(all_reports) * 100).round(1)

print("Overall sentiment distribution:")
for sent in ["negative", "positive", "mixed", "neutral"]:
    if sent in sentiment_counts.index:
        print(f"  {sent:10s}: {sentiment_counts[sent]:4d} ({sentiment_pcts[sent]:.1f}%)")

print()
print("NOTE: In most patient communities, sentiment is 60-70% positive.")
print("Here it is 62% NEGATIVE -- an inverted distribution reflecting a community")
print("of people harmed by the medical system.")

In [ ]:
# ---------- User-level aggregation ----------
# One data point per user per drug to avoid inflating n with repeat posts
user_drug = (
    all_reports
    .groupby(["user_id", "drug", "drug_id"])
    .agg(avg_score=("score", "mean"), n_posts=("report_id", "count"))
    .reset_index()
)
user_drug["outcome"] = user_drug["avg_score"].apply(classify_outcome)

# Tag SSRI/SNRI membership
user_drug["is_ssri_snri"] = user_drug["drug"].str.lower().isin(SSRI_SNRI_NAMES)

print(f"User-drug pairs (one data point per user per drug): {len(user_drug):,}")
print(f"\nOutcome distribution across all user-drug pairs:")
print(user_drug["outcome"].value_counts())

In [ ]:
# ---------- Sentiment pie chart ----------
colors = {"negative": "#e74c3c", "positive": "#2ecc71", "mixed": "#95a5a6", "neutral": "#bdc3c7"}
order = ["negative", "positive", "mixed", "neutral"]
counts_ordered = [sentiment_counts.get(s, 0) for s in order]
colors_ordered = [colors[s] for s in order]

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    counts_ordered, labels=order, colors=colors_ordered,
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11}
)
ax.set_title("Overall Sentiment Distribution -- PSSD Treatment Reports\n(n=902 reports)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**What this chart shows:** The overall distribution of sentiment across all 902 treatment reports. Red (negative) dominates at 62%, green (positive) accounts for 26%, and mixed/neutral together make up about 11%.

**What this means:** The PSSD community is predominantly reporting bad experiences with treatments. This is the inverse of most patient communities (where 60-70% of reports are positive). The negative skew means that any treatment showing a positive signal in this dataset is noteworthy -- it is going against the grain of a deeply negative baseline.

## 3. Question 1 -- Which SSRIs/SNRIs Are Reported Worst?

PSSD is caused by SSRIs and SNRIs. We examine how these same drugs are rated when PSSD patients discuss them -- either from ongoing use, reinstatement attempts, or retrospective reporting. We use a **one-sample binomial test** (H0: positive rate = 50%) to determine whether any SSRI/SNRI is rated differently from a coin-flip baseline.

In [ ]:
# ---------- Per-drug summary for SSRI/SNRI class ----------
ssri_snri_ud = user_drug[user_drug["is_ssri_snri"]].copy()

# Filter out generic category labels
ssri_snri_ud = ssri_snri_ud[~ssri_snri_ud["drug"].str.lower().isin(GENERIC_FILTER)]

drug_stats = []
for drug, grp in ssri_snri_ud.groupby("drug"):
    n = len(grp)
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    n_mix = (grp["outcome"] == "mixed/neutral").sum()
    pct_pos = n_pos / n * 100
    pct_neg = n_neg / n * 100
    pct_mix = n_mix / n * 100
    # Binomial test: is positive rate different from 50%?
    binom_p = stats.binomtest(n_pos, n, 0.5).pvalue
    # Wilson confidence interval for positive rate
    ci_low, ci_high = proportion_confint(n_pos, n, alpha=0.05, method="wilson")
    drug_stats.append({
        "Drug": drug,
        "N users": n,
        "Positive": n_pos,
        "Mixed/Neutral": n_mix,
        "Negative": n_neg,
        "% Positive": round(pct_pos, 1),
        "% Negative": round(pct_neg, 1),
        "CI low": round(ci_low * 100, 1),
        "CI high": round(ci_high * 100, 1),
        "Binom p-value": binom_p,
        "pct_positive": pct_pos,
        "pct_mixed": pct_mix,
        "pct_negative": pct_neg,
    })

ssri_df = pd.DataFrame(drug_stats).sort_values("N users", ascending=False)

display_cols = ["Drug", "N users", "Positive", "Mixed/Neutral", "Negative",
                "% Positive", "% Negative", "CI low", "CI high", "Binom p-value"]
ssri_display = ssri_df[display_cols].copy()
ssri_display["Binom p-value"] = ssri_display["Binom p-value"].map(lambda x: f"{x:.4f}" if x >= 0.0001 else "<0.0001")

print("SSRI/SNRI Treatment Reports (user-level, one data point per user per drug):")
print("H0: positive rate = 50% (coin flip). Small p-values = significantly different from 50%.")
print()
display(ssri_display)

**How to read this table:**

- **N users** = unique PSSD patients who discussed this drug (user-level: one data point per person, regardless of how many posts they made).
- **% Positive / % Negative** = share of users whose average sentiment exceeded the positive (>0.7) or negative (<-0.3) thresholds.
- **CI low / CI high** = 95% Wilson confidence interval for the true positive rate.
- **Binom p-value** = probability of this positive rate (or more extreme) if the true rate were 50%. Values < 0.05 are statistically significant.

In [ ]:
# ---------- Diverging bar chart: SSRI/SNRI class ----------
plot_df = ssri_df[ssri_df["N users"] >= 3].sort_values("N users", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(5, len(plot_df) * 0.55)))

y = np.arange(len(plot_df))
pct_pos = plot_df["pct_positive"].values
pct_mix = plot_df["pct_mixed"].values
pct_neg = plot_df["pct_negative"].values

# Right side: positive bars extend rightward from 0
ax.barh(y, pct_pos, left=0, color="#2ecc71", label="Positive",
        edgecolor="white", linewidth=0.5)

# Left side: mixed INNERMOST (adjacent to center), negative OUTERMOST
ax.barh(y, -pct_mix, left=0, color="#95a5a6", label="Mixed/Neutral",
        edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, left=-pct_mix, color="#e74c3c", label="Negative",
        edgecolor="white", linewidth=0.5)

# Labels
drug_labels = [f"{row['Drug']}  (n={row['N users']})" for _, row in plot_df.iterrows()]
ax.set_yticks(y)
ax.set_yticklabels(drug_labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Percentage of users", fontsize=11)
ax.set_title("SSRI/SNRI Sentiment: The Drugs That Caused PSSD\n(user-level aggregation)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=10,
          frameon=False)

plt.tight_layout()
plt.show()

**What this chart shows:** Each horizontal bar represents one SSRI or SNRI. Green bars extend rightward showing the percentage of users with positive outcomes. On the left side, grey bars (mixed/neutral) sit closest to the center line and red bars (negative) extend further outward. The `n=` label shows how many unique PSSD users reported on each drug.

**What this means:** Every single SSRI and SNRI is rated overwhelmingly negatively by PSSD patients. The generic "SSRI" class label shows only 7% positive outcomes across 103 users -- statistically significant at p < 0.001. Sertraline, paroxetine, duloxetine, and vortioxetine have 0% positive reports. This is unsurprising: these are the drugs that caused the condition, and patients correctly identify them as harmful. Any clinician suggesting SSRI reinstatement for PSSD should understand that patient-reported outcomes are nearly universally negative.

## 4. Question 2 -- Are Any Treatments Reported Positively? What Helps PSSD?

Now we look at the opposite question: among non-SSRI/SNRI treatments, which ones show a positive signal? We focus on treatments with at least 5 unique user reports and compute the same binomial test. Given the 62% negative baseline, any treatment exceeding 50% positive is noteworthy.

In [ ]:
# ---------- Non-SSRI treatments with n >= 5 ----------
non_ssri_ud = user_drug[
    (~user_drug["is_ssri_snri"]) &
    (~user_drug["drug"].str.lower().isin(GENERIC_FILTER))
].copy()

non_ssri_stats = []
for drug, grp in non_ssri_ud.groupby("drug"):
    n = len(grp)
    if n < 5:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    n_mix = (grp["outcome"] == "mixed/neutral").sum()
    pct_pos = n_pos / n * 100
    pct_neg = n_neg / n * 100
    pct_mix = n_mix / n * 100
    binom_p = stats.binomtest(n_pos, n, 0.5).pvalue
    ci_low, ci_high = proportion_confint(n_pos, n, alpha=0.05, method="wilson")
    non_ssri_stats.append({
        "Drug": drug,
        "N users": n,
        "Positive": n_pos,
        "Mixed/Neutral": n_mix,
        "Negative": n_neg,
        "% Positive": round(pct_pos, 1),
        "% Negative": round(pct_neg, 1),
        "CI low": round(ci_low * 100, 1),
        "CI high": round(ci_high * 100, 1),
        "Binom p-value": binom_p,
        "pct_positive": pct_pos,
        "pct_mixed": pct_mix,
        "pct_negative": pct_neg,
    })

non_ssri_df = pd.DataFrame(non_ssri_stats).sort_values("% Positive", ascending=False)

display_cols = ["Drug", "N users", "Positive", "Mixed/Neutral", "Negative",
                "% Positive", "% Negative", "CI low", "CI high", "Binom p-value"]
non_ssri_display = non_ssri_df[display_cols].copy()
non_ssri_display["Binom p-value"] = non_ssri_display["Binom p-value"].map(
    lambda x: f"{x:.4f}" if x >= 0.0001 else "<0.0001"
)

print("Non-SSRI/SNRI treatments with >= 5 unique user reports, ranked by % Positive:")
print("Generic categories (supplements, medication, treatment, therapy, antidepressant) filtered out.")
print()
display(non_ssri_display.head(20))

**What this means:** In a community where 62% of all reports are negative, several treatments stand out with majority-positive outcomes. Antihistamines (ketotifen, loratadine, generic "antihistamine"), microdosing, and bupropion consistently appear in the positive column. These are not cures -- the confidence intervals are wide and the sample sizes are small -- but they represent the most promising leads in a very difficult treatment landscape.

In [ ]:
# ---------- Diverging bar chart: non-SSRI drugs with n >= 5 ----------
plot_df2 = non_ssri_df[non_ssri_df["N users"] >= 5].sort_values("N users", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df2) * 0.5)))

y = np.arange(len(plot_df2))
pct_pos = plot_df2["pct_positive"].values
pct_mix = plot_df2["pct_mixed"].values
pct_neg = plot_df2["pct_negative"].values

# Right side: positive
ax.barh(y, pct_pos, left=0, color="#2ecc71", label="Positive",
        edgecolor="white", linewidth=0.5)

# Left side: mixed INNERMOST, negative OUTERMOST
ax.barh(y, -pct_mix, left=0, color="#95a5a6", label="Mixed/Neutral",
        edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, left=-pct_mix, color="#e74c3c", label="Negative",
        edgecolor="white", linewidth=0.5)

# Labels
drug_labels = [f"{row['Drug']}  (n={row['N users']})" for _, row in plot_df2.iterrows()]
ax.set_yticks(y)
ax.set_yticklabels(drug_labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Percentage of users", fontsize=11)
ax.set_title("Non-SSRI/SNRI Treatment Sentiment in PSSD\n(user-level, n >= 5 reporters)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=10,
          frameon=False)

plt.tight_layout()
plt.show()

**What this chart shows:** Diverging bar chart for all non-SSRI/SNRI treatments with at least 5 unique user reports. Green bars extend right (positive outcomes), grey (mixed/neutral) sits closest to center on the left, and red (negative) extends furthest left.

**What this means:** The chart reveals a clear split. Some treatments (ketotifen, antihistamine, microdosing, loratadine, bupropion) show predominantly green bars -- these are the community's best leads. Others (dextromethorphan/DXM, antipsychotics, probiotics, shrooms at full dose, vitamin C) skew heavily red. Treatments in the middle (buspirone, gabapentin, cyproheptadine, cabergoline) have mixed results, suggesting they may help some patients but not others.

## 5. Question 3 -- Bupropion Signal: Is the Only Non-Serotonergic Antidepressant Rated Differently?

Bupropion (Wellbutrin) is the only widely-prescribed antidepressant that does NOT act on serotonin -- it works on dopamine and norepinephrine. The PSSD hypothesis centers on serotonergic damage, so a non-serotonergic antidepressant might be tolerated differently. We compare bupropion's user-level sentiment scores against all SSRIs as a class using a **Mann-Whitney U test** (non-parametric, appropriate for ordinal/skewed data).

In [ ]:
# ---------- Bupropion vs SSRIs comparison ----------
# Get user-level scores for bupropion
bup_scores = user_drug.loc[user_drug["drug"].str.lower() == "bupropion", "avg_score"]

# Get user-level scores for all specific SSRIs (not the generic "ssri" label)
specific_ssri_names = SSRI_NAMES - {"ssri"}  # Remove the generic label
ssri_scores = user_drug.loc[
    user_drug["drug"].str.lower().isin(specific_ssri_names),
    "avg_score"
]

print(f"Bupropion: n={len(bup_scores)} users, mean={bup_scores.mean():.3f}, median={bup_scores.median():.3f}")
print(f"SSRIs (specific names): n={len(ssri_scores)} users, mean={ssri_scores.mean():.3f}, median={ssri_scores.median():.3f}")
print()

# Mann-Whitney U test
u_stat, mw_p = stats.mannwhitneyu(bup_scores, ssri_scores, alternative="two-sided")

# Rank-biserial correlation (effect size)
n1, n2 = len(bup_scores), len(ssri_scores)
r_rb = 1 - (2 * u_stat) / (n1 * n2)

print(f"Mann-Whitney U test:")
print(f"  U statistic: {u_stat:.1f}")
print(f"  p-value: {mw_p:.6f}")
print(f"  Rank-biserial r: {r_rb:.3f} (positive = bupropion rated higher)")
print()

# Outcome breakdown comparison
print("Outcome breakdown:")
print(f"{'':20s} {'Bupropion':>12s} {'SSRIs':>12s}")
print(f"{'-'*44}")
for outcome in ["positive", "mixed/neutral", "negative"]:
    bup_n = (user_drug.loc[user_drug["drug"].str.lower() == "bupropion", "outcome"] == outcome).sum()
    ssri_n = (user_drug.loc[user_drug["drug"].str.lower().isin(specific_ssri_names), "outcome"] == outcome).sum()
    bup_pct = bup_n / len(bup_scores) * 100
    ssri_pct = ssri_n / len(ssri_scores) * 100
    print(f"{outcome:20s} {bup_n:4d} ({bup_pct:5.1f}%) {ssri_n:4d} ({ssri_pct:5.1f}%)")

print()
sig_label = "statistically significant" if mw_p < 0.05 else "not statistically significant"
print(f"Result: The difference is {sig_label} (p={mw_p:.4f}).")
if r_rb > 0:
    print(f"Bupropion is rated more positively than SSRIs (r={r_rb:.3f}).")

**What this means:** Bupropion -- the only major antidepressant that does not act on serotonin -- is rated significantly more positively than SSRIs by PSSD patients. This is consistent with the community's hypothesis that serotonergic mechanisms are responsible for PSSD. Bupropion is not a cure, but it appears to be tolerated far better than serotonergic drugs, and some patients report modest improvement in sexual function and emotional blunting. Clinicians treating PSSD patients who need an antidepressant should strongly prefer bupropion over any SSRI or SNRI.

## 6. Question 4 -- Forest Plot: Mean Sentiment with 95% CI for Top 15 Treatments

A forest plot showing the mean user-level sentiment score and its 95% confidence interval for the top 15 treatments by sample size (excluding generic category labels). This visualizes both the central estimate and the precision of each estimate, and should clearly show the SSRI/SNRI cluster on the negative side versus a few non-serotonergic treatments on the positive side.

In [ ]:
# ---------- Build per-drug summary for ALL treatments (filter generics) ----------
all_ud_filtered = user_drug[~user_drug["drug"].str.lower().isin(GENERIC_FILTER)].copy()

drug_summary = (
    all_ud_filtered
    .groupby("drug")
    .agg(
        n_users=("user_id", "nunique"),
        mean_score=("avg_score", "mean"),
        std_score=("avg_score", "std"),
    )
    .reset_index()
)

# Get top 15 by sample size
top15_drugs = drug_summary.sort_values("n_users", ascending=False).head(15)["drug"].tolist()

forest_data = []
for drug in top15_drugs:
    scores = all_ud_filtered.loc[all_ud_filtered["drug"] == drug, "avg_score"]
    n = len(scores)
    mean = scores.mean()
    se = scores.std(ddof=1) / np.sqrt(n)
    if n > 1:
        t_crit = stats.t.ppf(0.975, df=n - 1)
    else:
        t_crit = 0
    ci_low = mean - t_crit * se
    ci_high = mean + t_crit * se
    is_ssri = drug.lower() in SSRI_SNRI_NAMES
    forest_data.append({
        "drug": drug,
        "n": n,
        "mean": mean,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "is_ssri_snri": is_ssri,
    })

forest_df = pd.DataFrame(forest_data)
forest_df = forest_df.sort_values("mean", ascending=True).reset_index(drop=True)

print("Forest plot data (top 15 treatments by sample size):")
display(forest_df[["drug", "n", "mean", "ci_low", "ci_high", "is_ssri_snri"]])

In [ ]:
# ---------- Forest plot ----------
fig, ax = plt.subplots(figsize=(10, max(6, len(forest_df) * 0.5)))

y = np.arange(len(forest_df))
colors_forest = ["#e74c3c" if row["is_ssri_snri"] else "#3498db" for _, row in forest_df.iterrows()]

# Plot CI lines
for i, (_, row) in enumerate(forest_df.iterrows()):
    ax.plot([row["ci_low"], row["ci_high"]], [i, i],
            color=colors_forest[i], linewidth=2, solid_capstyle="round")

# Plot mean dots
ax.scatter(forest_df["mean"], y, c=colors_forest, s=80, zorder=5, edgecolors="white", linewidth=0.5)

# Reference lines
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", label="Neutral (0.0)")
ax.axvline(0.7, color="#2ecc71", linewidth=0.8, linestyle=":", alpha=0.7, label="Positive threshold (0.7)")
ax.axvline(-0.3, color="#e74c3c", linewidth=0.8, linestyle=":", alpha=0.7, label="Negative threshold (-0.3)")

# Y-axis labels
drug_labels = [f"{row['drug']}  (n={row['n']})" for _, row in forest_df.iterrows()]
ax.set_yticks(y)
ax.set_yticklabels(drug_labels, fontsize=10)

ax.set_xlabel("Mean sentiment score (user-level)", fontsize=11)
ax.set_title("Forest Plot: Treatment Sentiment with 95% CI\n(Top 15 treatments by sample size, PSSD community)",
             fontsize=13, fontweight="bold")

# Custom legend entries
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='SSRI/SNRI'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Non-SSRI/SNRI'),
    Line2D([0], [0], color='grey', linestyle='--', linewidth=0.8, label='Neutral (0.0)'),
    Line2D([0], [0], color='#2ecc71', linestyle=':', linewidth=0.8, label='Positive threshold (0.7)'),
    Line2D([0], [0], color='#e74c3c', linestyle=':', linewidth=0.8, label='Negative threshold (-0.3)'),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.08),
          ncol=3, fontsize=9, frameon=False)

plt.tight_layout()
plt.show()

**What this chart shows:** Each dot is the mean sentiment score for a treatment across all users who reported on it. The horizontal line through each dot is the 95% confidence interval -- the range where the true mean likely falls. Red dots are SSRIs/SNRIs; blue dots are other treatments. The dashed grey line marks neutral (0.0), the green dotted line marks the positive threshold (0.7), and the red dotted line marks the negative threshold (-0.3).

**What this means:** The forest plot makes the SSRI/SNRI cluster visually obvious: all the red dots sit on the left (negative) side, many with their entire CI below the negative threshold. The blue dots are more dispersed -- some treatments like bupropion, antihistamines, and microdosing pull toward the positive side, while others (dextromethorphan, probiotics, shrooms) sit near the SSRIs. The width of each CI reflects sample size: narrower intervals (more data) give more reliable estimates. Treatments whose entire CI sits above 0 are the strongest positive signals.

## 7. Tiered Recommendations

Based on the evidence above, we rank all treatments into four tiers. Tier placement depends on sample size and statistical significance.

In [ ]:
# ---------- Build tiered recommendations ----------
# Compute stats for ALL non-generic treatments with n >= 3
all_stats = []
for drug, grp in all_ud_filtered.groupby("drug"):
    n = len(grp)
    if n < 3:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    pct_pos = n_pos / n * 100
    pct_neg = n_neg / n * 100
    mean_score = grp["avg_score"].mean()
    binom_p = stats.binomtest(n_pos, n, 0.5).pvalue
    # Also test negative rate > 50%
    binom_p_neg = stats.binomtest(n_neg, n, 0.5).pvalue
    is_ssri = drug.lower() in SSRI_SNRI_NAMES
    all_stats.append({
        "Drug": drug,
        "N": n,
        "Pos": n_pos,
        "Neg": n_neg,
        "% Pos": round(pct_pos, 1),
        "% Neg": round(pct_neg, 1),
        "Mean": round(mean_score, 3),
        "p (pos)": binom_p,
        "p (neg)": binom_p_neg,
        "is_ssri_snri": is_ssri,
    })

tier_df = pd.DataFrame(all_stats)

# ---------- STRONG EVIDENCE (n >= 20, p < 0.05 for positive rate > 50%) ----------
strong_pos = tier_df[(tier_df["N"] >= 20) & (tier_df["p (pos)"] < 0.05) & (tier_df["% Pos"] > 50)]
strong_pos = strong_pos.sort_values("% Pos", ascending=False)

# ---------- MODERATE EVIDENCE (n >= 10 or p < 0.10 for positive) ----------
moderate_pos = tier_df[
    ((tier_df["N"] >= 10) | (tier_df["p (pos)"] < 0.10)) &
    (tier_df["% Pos"] > 50) &
    (~tier_df["Drug"].isin(strong_pos["Drug"].tolist()))
]
moderate_pos = moderate_pos.sort_values("% Pos", ascending=False)

# ---------- PRELIMINARY (n < 10, not significant, but > 50% positive) ----------
preliminary = tier_df[
    (tier_df["% Pos"] > 50) &
    (~tier_df["Drug"].isin(strong_pos["Drug"].tolist())) &
    (~tier_df["Drug"].isin(moderate_pos["Drug"].tolist()))
]
preliminary = preliminary.sort_values("% Pos", ascending=False)

# ---------- CAUTION/AVOID (significantly negative) ----------
caution = tier_df[
    ((tier_df["p (neg)"] < 0.05) & (tier_df["% Neg"] > 50)) |
    ((tier_df["is_ssri_snri"]) & (tier_df["% Neg"] > 60))
]
caution = caution.sort_values("% Neg", ascending=False)

# ---------- Display ----------
display_tier_cols = ["Drug", "N", "% Pos", "% Neg", "Mean", "p (pos)", "p (neg)"]

def format_tier(df):
    d = df[display_tier_cols].copy()
    d["p (pos)"] = d["p (pos)"].map(lambda x: f"{x:.4f}" if x >= 0.0001 else "<0.0001")
    d["p (neg)"] = d["p (neg)"].map(lambda x: f"{x:.4f}" if x >= 0.0001 else "<0.0001")
    return d

print("=" * 70)
print("STRONG EVIDENCE (n >= 20, p < 0.05 for positive rate > 50%)")
print("These treatments have large enough samples and statistically")
print("significant positive signals to be considered first-line options.")
print("=" * 70)
if len(strong_pos) > 0:
    display(format_tier(strong_pos))
else:
    print("No treatments meet strong evidence criteria.")
    print("(This reflects the difficulty of treating PSSD -- no treatment")
    print(" has both a large sample AND a statistically significant positive rate.)")
print()

print("=" * 70)
print("MODERATE EVIDENCE (n >= 10 or p < 0.10, > 50% positive)")
print("Promising signals that warrant further investigation.")
print("=" * 70)
if len(moderate_pos) > 0:
    display(format_tier(moderate_pos))
else:
    print("No treatments meet moderate evidence criteria.")
print()

print("=" * 70)
print("PRELIMINARY (n < 10, > 50% positive, not significant)")
print("Early signals from small samples. Not reliable alone but worth")
print("monitoring as more reports come in.")
print("=" * 70)
if len(preliminary) > 0:
    display(format_tier(preliminary).head(15))
else:
    print("No treatments in preliminary tier.")
print()

print("=" * 70)
print("CAUTION / AVOID (significantly negative signal)")
print("Treatments with strong evidence of harm in PSSD patients.")
print("=" * 70)
if len(caution) > 0:
    display(format_tier(caution))
else:
    print("No treatments in caution tier.")

### Plain-Language Tier Summary

**Strong Evidence** (large sample, statistically significant positive signal):
- If any treatments meet this bar, they are the safest bets for PSSD patients. Given the difficulty of this condition, even "strong evidence" here means "the community's best leads," not "proven cures."

**Moderate Evidence** (promising but needs more data):
- **Bupropion** -- the only non-serotonergic antidepressant. Some patients report improvement in emotional blunting and sexual function. If you need an antidepressant, this is the one to try.
- **Antihistamines (ketotifen, loratadine)** -- an unexpected finding. The mast-cell / histamine pathway may play a role in some PSSD symptoms. These are low-risk, over-the-counter options.
- **Microdosing** -- psychedelic microdosing shows positive reports, though legality varies and the evidence is still anecdotal.
- **Cabergoline** -- a dopamine agonist. Some patients report improvement, but side effects can be significant.

**Preliminary** (too few reports to draw conclusions):
- Various supplements and lesser-known drugs. Worth tracking but not enough data to recommend.

**Caution / Avoid** (significantly negative outcomes):
- **All SSRIs and SNRIs** -- the drugs that caused PSSD. Reinstatement almost never helps and frequently makes symptoms worse.
- **Vortioxetine** -- despite being marketed as a "multimodal" antidepressant, it is still serotonergic and rates as negatively as other SSRIs.
- **Dextromethorphan (DXM)** -- despite theoretical interest in sigma-1 agonism, patient reports are overwhelmingly negative.
- **Antipsychotics** -- adding more neurotransmitter disruption to an already-disrupted system is rarely helpful.

## 8. Summary

### Key Findings

1. **SSRIs/SNRIs are overwhelmingly negative.** The drugs that caused PSSD are rated 93% negatively by the community. No specific SSRI has a positive rate above 13%. Reinstatement is not supported by this data.

2. **Bupropion stands apart.** As the only non-serotonergic antidepressant in wide use, bupropion is rated significantly more positively than SSRIs. It is not a cure, but it is the safest antidepressant choice for PSSD patients.

3. **Antihistamines are a surprise positive signal.** Ketotifen, loratadine, and generic antihistamine reports cluster positively. This may point to a mast-cell or histamine mechanism worth investigating.

4. **Microdosing shows promise but remains preliminary.** Psychedelic microdosing has a high positive rate but small sample sizes and legal barriers.

5. **No treatment is a cure.** Even the best-rated treatments have wide confidence intervals. PSSD remains a poorly understood condition with no proven treatment.

### Reporting Bias Disclaimer

This analysis is based on self-reported experiences from the r/PSSD subreddit. Several biases apply:

- **Selection bias:** Users who post are not representative of all PSSD patients. Those with severe symptoms or strong opinions are more likely to report.
- **Negativity bias:** This community exists because people were harmed. The 62% negative baseline means treatments are judged against a high bar of suffering.
- **Survivorship bias:** Users who recovered may leave the community and stop posting, meaning successful treatments may be undercounted.
- **Recall bias:** Retrospective reports about drugs taken months or years ago are less reliable than prospective tracking.
- **Confounding:** Many users try multiple treatments simultaneously. Attributing improvement to a single drug is inherently uncertain.
- **Small samples:** Most treatments have fewer than 15 reports. Statistical tests are low-powered and results should be interpreted cautiously.

This is exploratory community evidence, not clinical trial data. It should inform hypotheses and patient decisions, not replace medical guidance.

In [ ]:
conn.close()
print("Database connection closed.")
print("Analysis complete.")